###imports

In [1]:
!pip install lets-plot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

In [3]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from lets_plot import *
from lets_plot.mapping import as_discrete

In [4]:
LetsPlot.setup_html()

In [5]:
df_department = pd.read_csv("/content/department-v2.csv")
df_seniority = pd.read_csv("/content/seniority-v2.csv")

###data


In [6]:
def workers_positions_to_csv(data):
    people = json.loads(Path(data).read_text(encoding="utf-8"))
    rows = []

    for person_idx, jobs in enumerate(people, start=1):
        jobs_sorted = sorted(jobs, key=lambda d: (d.get("startDate") is None, d.get("startDate")))
        for job_idx, j in enumerate(jobs_sorted, start=1):
            rows.append({
                "person_id": person_idx,
                "job_no": job_idx,
                "organization": j.get("organization"),
                "linkedin": j.get("linkedin"),
                "position": j.get("position"),
                "startDate": j.get("startDate"),
                "endDate": j.get("endDate"),
                "status": j.get("status"),
                "department": j.get("department"),
                "seniority": j.get("seniority"),
            })

    df = pd.DataFrame(rows, columns=[
        "person_id","job_no","organization","linkedin","position",
        "startDate","endDate","status","department","seniority"
    ])
    df["person_id"] = df["person_id"].astype(int)
    df["job_no"] = df["job_no"].astype(int)
    return df

In [7]:
df_t_annotated= workers_positions_to_csv("/content/linkedin-cvs-annotated.json")
df_t_notannotated= workers_positions_to_csv("/content/linkedin-cvs-not-annotated.json")

In [8]:
df_pred_bow = pd.read_csv("/content/df_t_notannotated_pred_bow_both_threshold.csv")
df_pred_emb = pd.read_csv("/content/df_t_notannotated_pred_embeddings_both_threshold.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/df_t_notannotated_pred_bow_both_threshold.csv'

In [9]:

DEPT_PALETTE = {
    "Marketing": "#b3de69",
    "Sales": "#fb8072",
    "Information Technology": "#80b1d3",
    "Business Development": "#fdb462",
    "Project Management": "#bebada",
    "Consulting": "#bc80bd",
    "Administrative": "#ccebc5",
    "Customer Support": "#fccde5",
    "Human Resources": "#ffed6f",
    "Purchasing": "#8dd3c7",
    "Other": "#d9d9d9",
}

SENIORITY_PALETTE = {
    "Junior": "#66c2a5",
    "Professional": "#a6d854",
    "Senior": "#fc8d62",
    "Lead": "#8da0cb",
    "Management": "#e78ac3",
    "Director": "#ffd92f",
}


#insight

###general

In [ ]:
for df_name, df_obj in [("department-v2", df_department), ("seniority-v2", df_seniority)]:
    print(df_name)
    print("--------------------------------------------------------")
    print(df_obj.describe())
    print("\n")

for df_name, df_obj in [("linkedin-cvs-annotated", df_t_annotated), ("linkedin-cvs-not-annotated", df_t_notannotated)]:
    print(df_name)
    print("--------------------------------------------------------")
    print(df_obj.info())
    print("\n")

department-v2
--------------------------------------------------------
                      text      label
count                10145      10145
unique               10145         11
top     VP Sales Processes  Marketing
freq                     1       4295


seniority-v2
--------------------------------------------------------
                      text   label
count                 9428    9428
unique                9428       5
top     VP Sales Processes  Senior
freq                     1    3733


linkedin-cvs-annotated
--------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2638 entries, 0 to 2637
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   person_id     2638 non-null   int64 
 1   job_no        2638 non-null   int64 
 2   organization  2638 non-null   object
 3   linkedin      2638 non-null   object
 4   position      2638 non-null   object
 5   star

##Label distrubution

(absolute numbers) samples per class

In [ ]:
print("department")
print("--------------------------------------------------------")
print('Value counts for department-v2["label"]:')
print(df_department["label"].value_counts())

print('\nValue counts for linkedin-cvs-annotated["department"]:')
print(df_t_annotated["department"].value_counts())

print("\n")
print("seniority")
print("--------------------------------------------------------")

print('\nValue counts for seniority-v2["label"]:')
print(df_seniority["label"].value_counts())

print('\nValue counts for linkedin-cvs-annotated["seniority"]:')
print(df_t_annotated["seniority"].value_counts())

department
--------------------------------------------------------
Value counts for department-v2["label"]:
label
Marketing                 4295
Sales                     3328
Information Technology    1305
Business Development       620
Project Management         201
Consulting                 167
Administrative              83
Other                       42
Purchasing                  40
Customer Support            33
Human Resources             31
Name: count, dtype: int64

Value counts for linkedin-cvs-annotated["department"]:
department
Other                     1252
Information Technology     312
Sales                      219
Consulting                 195
Project Management         175
Marketing                  133
Administrative              84
Business Development        78
Purchasing                  72
Human Resources             70
Customer Support            48
Name: count, dtype: int64


seniority
--------------------------------------------------------

Value counts f

In [ ]:
seniority_counts = df_t_annotated["seniority"].value_counts().reset_index()
department_counts = df_t_annotated["department"].value_counts().reset_index()

Distribution_Seniority_linkedin_cvs_annotated = (ggplot(seniority_counts) +
      geom_bar(aes(x="seniority", y="count", fill="seniority"), stat="identity") +
      #geom_text(aes(x="seniority", y="count", label="count"),va="bottom", nudge_y=10, size=7) +
      ggtitle("Distribution of Seniority - linkedin-cvs-annotated") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=SENIORITY_PALETTE)
      )
Distribution_Departments_linkedin_cvs_annotated = (ggplot(department_counts) +
      geom_bar(aes(x="department", y="count", fill="department"), stat="identity") +
      geom_text(aes(x="department", y="count", label="count"),va="bottom", nudge_y=10, size=7) +
      ggtitle("Distribution of Departments - linkedin-cvs-annotated") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=DEPT_PALETTE)
      )

seniority_counts = df_seniority["label"].value_counts().reset_index()
department_counts = df_department["label"].value_counts().reset_index()

Distribution_Seniority_seniority_v2 = (ggplot(seniority_counts) +
      geom_bar(aes(x="label", y="count", fill="label"), stat="identity") +
      #geom_text(aes(x="label", y="count", label="count"),va="bottom", nudge_y=20, size=7) +
      ggtitle("Distribution of Seniority - seniority-v2") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=SENIORITY_PALETTE)
      )
Distribution_Departments_department_v2 = (ggplot(department_counts) +
      geom_bar(aes(x="label", y="count", fill="label"), stat="identity") +
      geom_text(aes(x="label", y="count", label="count"),va="bottom", nudge_y=50, size=7) +
      ggtitle("Distribution of Departments - department-v2") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=DEPT_PALETTE)
      )

Distribution_Departments_linkedin_cvs_annotated.show()
Distribution_Departments_department_v2.show()

Distribution_Seniority_linkedin_cvs_annotated.show()
Distribution_Seniority_seniority_v2.show()



In [ ]:
#Distribution_Departments_department_v2.to_png("Distribution_Departments_department_v2.png")
#Distribution_Departments_linkedin_cvs_annotated.to_png("Distribution_Departments_linkedin_cvs_annotated.png")
#Distribution_Seniority_linkedin_cvs_annotated.to_png("Distribution_Seniority_linkedin_cvs_annotated.png")
#Distribution_Seniority_seniority_v2.to_png("Distribution_Seniority_seniority_v2.png")

In [ ]:
seniority_counts = df_pred_bow["seniority"].value_counts().reset_index()
department_counts = df_pred_bow["department"].value_counts().reset_index()

Distribution_Seniority_linkedin_df_pred_bow = (ggplot(seniority_counts) +
      geom_bar(aes(x="seniority", y="count", fill="seniority"), stat="identity") +
      #geom_text(aes(x="seniority", y="count", label="count"),va="bottom", nudge_y=10, size=7) +
      ggtitle("Distribution of Seniority \n - linkedin-not-annotated-cvs predicted with Bag of Words (TH)") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=SENIORITY_PALETTE)
      )
Distribution_Departments_linkedin_df_pred_bow = (ggplot(department_counts) +
      geom_bar(aes(x="department", y="count", fill="department"), stat="identity") +
      geom_text(aes(x="department", y="count", label="count"),va="bottom", nudge_y=10, size=7) +
      ggtitle("Distribution of Departments \n - linkedin-not-annotated-cvs predicted with Bag of Words (TH)") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=DEPT_PALETTE)
      )

Distribution_Departments_linkedin_df_pred_bow.show()
Distribution_Seniority_linkedin_df_pred_bow.show()

In [ ]:
seniority_counts = df_pred_emb["seniority"].value_counts().reset_index()
department_counts = df_pred_emb["department"].value_counts().reset_index()

Distribution_Seniority_linkedin_df_pred_emb = (ggplot(seniority_counts) +
      geom_bar(aes(x="seniority", y="count", fill="seniority"), stat="identity") +
      #geom_text(aes(x="seniority", y="count", label="count"),va="bottom", nudge_y=10, size=7) +
      ggtitle("Distribution of Seniority \n - linkedin-not-annotated-cvs predicted with embeddings (TH)") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=SENIORITY_PALETTE)
      )
Distribution_Departments_linkedin_df_pred_emb = (ggplot(department_counts) +
      geom_bar(aes(x="department", y="count", fill="department"), stat="identity") +
      geom_text(aes(x="department", y="count", label="count"),va="bottom", nudge_y=10, size=7) +
      ggtitle("Distribution of Departments \n  - linkedin-not-annotated-cvs predicted with embeddings (TH)") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=DEPT_PALETTE)
      )

Distribution_Departments_linkedin_df_pred_emb.show()
Distribution_Seniority_linkedin_df_pred_emb.show()

In [ ]:
#Distribution_Departments_linkedin_df_pred_bow.to_png("Distribution_Departments_linkedin_df_pred_bow.png")
#Distribution_Seniority_linkedin_df_pred_bow.to_png("Distribution_Seniority_linkedin_df_pred_bow.png")

In [ ]:
#Distribution_Departments_linkedin_df_pred_emb.to_png("Distribution_Departments_linkedin_df_pred_emb.png")
#Distribution_Seniority_linkedin_df_pred_emb.to_png("Distribution_Seniority_linkedin_df_pred_emb.png")

In [ ]:
df_pred_bow = pd.read_csv("/content/df_t_notannotated_pred_bow_no_thresholdv.csv")
df_pred_emb = pd.read_csv("/content/df_t_notannotated_pred_embeddings_no_threshold.csv")

In [ ]:
seniority_counts = df_pred_bow["seniority"].value_counts().reset_index()
department_counts = df_pred_bow["department"].value_counts().reset_index()

Distribution_Seniority_linkedin_df_pred_bow = (ggplot(seniority_counts) +
      geom_bar(aes(x="seniority", y="count", fill="seniority"), stat="identity") +
      #geom_text(aes(x="seniority", y="count", label="count"),va="bottom", nudge_y=10, size=7) +
      ggtitle("Distribution of Seniority \n - linkedin-not-annotated-cvs predicted with Bag of Words") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=SENIORITY_PALETTE)
      )
Distribution_Departments_linkedin_df_pred_bow = (ggplot(department_counts) +
      geom_bar(aes(x="department", y="count", fill="department"), stat="identity") +
      geom_text(aes(x="department", y="count", label="count"),va="bottom", nudge_y=10, size=7) +
      ggtitle("Distribution of Departments \n - linkedin-not-annotated-cvs predicted with Bag of Words") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=DEPT_PALETTE)
      )

Distribution_Departments_linkedin_df_pred_bow.show()
Distribution_Seniority_linkedin_df_pred_bow.show()

In [ ]:
seniority_counts = df_pred_emb["seniority"].value_counts().reset_index()
department_counts = df_pred_emb["department"].value_counts().reset_index()

Distribution_Seniority_linkedin_df_pred_emb = (ggplot(seniority_counts) +
      geom_bar(aes(x="seniority", y="count", fill="seniority"), stat="identity") +
      #geom_text(aes(x="seniority", y="count", label="count"),va="bottom", nudge_y=10, size=7) +
      ggtitle("Distribution of Seniority \n - linkedin-not-annotated-cvs predicted with embeddings") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=SENIORITY_PALETTE)
      )
Distribution_Departments_linkedin_df_pred_emb = (ggplot(department_counts) +
      geom_bar(aes(x="department", y="count", fill="department"), stat="identity") +
      geom_text(aes(x="department", y="count", label="count"),va="bottom", nudge_y=10, size=7) +
      ggtitle("Distribution of Departments \n  - linkedin-not-annotated-cvs predicted with embeddings") +
      theme_minimal() +
      theme(axis_text_x=element_text(angle=45, hjust=1)) +
      #scale_fill_brewer(palette = "Set2") +
      theme(legend_position="none") +
      scale_fill_manual(values=DEPT_PALETTE)
      )

Distribution_Departments_linkedin_df_pred_emb.show()
Distribution_Seniority_linkedin_df_pred_emb.show()

##career history

“How many departments per person?”: number of unique departments per person

In [22]:
dept_per_person = df_t_annotated.groupby("person_id")["department"].nunique().reset_index(name="unique_departments")
dept_distribution = dept_per_person["unique_departments"].value_counts().sort_index()


df_plot = dept_distribution.reset_index()
df_plot.columns = ["unique_departments", "n_people"]

departments_per_person = (ggplot(df_plot) +
    geom_bar(aes(x="unique_departments", y="n_people"), stat="identity", fill="#08589e") +
    labs(title="Number of Departments per Person - linkedin-cvs-annotated", x="Unique Departments", y="Number of People") +
    theme_minimal()
    )
departments_per_person

In [23]:
#departments_per_person.to_png("departments_per_person.png")

“Seniority increases over time?” (checks per person if it never decreases over time)

In [24]:
# maybe change ranking
seniority_rank = {
    "Junior": 1,
    "Professional": 2,
    "Senior": 3,
    "Lead": 4,
    "Management": 5,
    "Director": 6
}

df_sen = df_t_annotated.dropna(subset=["person_id", "startDate", "seniority"]).copy() # drop those with start date na
df_sen["seniority_level"] = df_sen["seniority"].map(seniority_rank)
df_sen["startDate"] = pd.to_datetime(df_sen["startDate"], errors="coerce")

def is_increasing(s):
    return s.is_monotonic_increasing

seniority_trend = df_sen.sort_values(["person_id", "startDate"]).groupby("person_id")["seniority_level"].apply(is_increasing)

print("Seniority trend (always increasing?):")
print(seniority_trend.value_counts())

df_trend = seniority_trend.reset_index()

seniority_always_increasing = (ggplot(df_trend) +
    geom_bar(aes(x="seniority_level")) +
    labs(title="Do People Increase in Seniority Over Time? - linkedin-cvs-annotated",
         x="Always Increasing?",
         y="Number of People") +
    theme_minimal() +
    scale_fill_brewer(palette = "Set2")
    )
seniority_always_increasing.show()


Seniority trend (always increasing?):
seniority_level
True     272
False    240
Name: count, dtype: int64


In [25]:
#seniority_always_increasing.to_png("seniority_always_increasing.png")

“always increasing” but within one department

In [26]:
dept_sen_trend = df_sen.groupby(["person_id", "department"])["seniority_level"].apply(is_increasing).reset_index(name="is_increasing")

print(dept_sen_trend["is_increasing"].value_counts())


seniority_increasing_same_department = (ggplot(dept_sen_trend) +
    geom_bar(aes(x="is_increasing")) +
    labs(title="Career Progression Within Same Department - linkedin-cvs-annotated",
         x="Always Increasing?",
         y="Number of Department Histories") +
    theme_minimal() +
    scale_fill_brewer(palette = "Set2")
    )
seniority_increasing_same_department.show()


is_increasing
True     898
False    193
Name: count, dtype: int64


In [27]:
#seniority_increasing_same_department.to_png("seniority_increasing_same_department.png")

Example career timelines

In [28]:
person_counts = df_sen["person_id"].value_counts()
filtered_person_ids_series = person_counts[person_counts > 2].index

sample_ids = filtered_person_ids_series.to_series().sample(n=16, replace=False, random_state=42).tolist()

df_sample = df_sen[df_sen["person_id"].isin(sample_ids)].copy()

df_sample["startDate"] = pd.to_datetime(df_sample["startDate"], errors="coerce")

career_paths = (
    ggplot(df_sample) +
    geom_point(aes(x="startDate", y="seniority_level", color="person_id"), size=2, alpha=0.7) +
    geom_smooth(aes(x="startDate", y="seniority_level", group="person_id", color="person_id"), method="lm", se=False, size=1) +
    scale_y_continuous(breaks=list(range(1, 7)), labels=["Junior", "Professional", "Senior", "Lead", "Management", "Director"]) +
    labs(title="Career Progression for Sample People - linkedin-cvs-annotated",
        x="Time",
        y="Seniority Level",
        color="Person ID") +
    ggsize(1000, 800) +
    theme_minimal() +
    facet_wrap("person_id") +
    scale_color_brewer(palette="Set2") +
    theme(legend_position="none")
    )

career_paths.show()

In [29]:
#career_paths.to_png("career_paths.png")

“always increasing” but person staying in only one department

In [30]:
unique_depts = df_sen.groupby("person_id")["department"].nunique().reset_index(name="unique_departments")
one_dept_promos = dept_sen_trend.groupby("person_id")["is_increasing"].max().reset_index(name="got_promoted")
merged = unique_depts.merge(one_dept_promos, on="person_id")

only_one_dept = merged[merged["unique_departments"] == 1]

seniority_only_one_department_increasing = (ggplot(only_one_dept) +
    geom_bar(aes(x="got_promoted"), fill="#e76f51") +
    labs(title="People in only One Department Who Got Promoted - linkedin-cvs-annotated", x="Promoted?", y="Number of People") +
    theme_minimal() +
    scale_fill_brewer(palette = "Set2")
    )
seniority_only_one_department_increasing.show()

In [31]:
#seniority_only_one_department_increasing.to_png("seniority_only_one_department_increasing.png")

In [32]:
sample_ids = df_sen["person_id"].value_counts().index.tolist()
df_sample = df_sen[df_sen["person_id"].isin(sample_ids)]

career_progression_all_people = (ggplot(df_sample) +
    geom_line(aes(x="startDate", y="seniority_level", group="person_id"), color="#08589e", size=2, alpha=0.07) +
    geom_point(aes(x="startDate", y="seniority_level"), color="#08589e", size=1, alpha=0.1) +
    labs(title="Career Progression for All People - linkedin-cvs-annotated", x="Time", y="Seniority Level") +
    scale_y_continuous(breaks=list(range(1, 7)), labels=["Junior", "Professional", "Senior", "Lead", "Management", "Director"]) +
    theme_minimal()
    )
career_progression_all_people.show()

In [33]:
#career_progression_all_people.to_png("career_progression_all_people.png")

'/content/career_progression_all_people.png'

In [44]:
seniority_over_time = (
    ggplot(df_sen, aes(x="startDate", y="seniority_level")) +
    geom_point(position=position_jitter(width=30, height=0.1), alpha=0.25, size=1) +
    geom_smooth(method="lm", se=False, size=1.1, color="red") +
    scale_y_continuous(breaks=[1,2,3,4,5,6], labels=["Junior","Professional","Senior","Lead","Management","Director"]) +
    labs(title="Seniority over time - linkedin-cvs-annotated", x="Start date", y="Seniority level") +
    theme_minimal()
)

seniority_over_time


In [45]:
#seniority_over_time.to_png("seniority_over_time.png")

'/content/seniority_over_time.png'

##Position/Text

###job titles -most common words

####**linkedin-cvs-annotated - department**

#####Top unigrams per department

In [12]:
TOP_K = 11
NGRAMS = (1, 1)

df = df_t_annotated.copy()
df["pos"] = df["position"]

STOP = []
STOP = ["and", "of", "in", "for", "der", "und", "für"]

vec_annotated = CountVectorizer(lowercase=True, min_df=2, ngram_range=NGRAMS, stop_words=STOP)
X_annotated = vec_annotated.fit_transform(df["pos"])

terms_annotated = np.array(vec_annotated.get_feature_names_out())
overall_annotated = np.asarray(X_annotated.sum(axis=0)).ravel()

words_top_annotated = (pd.DataFrame({"term": terms_annotated, "count": overall_annotated}).sort_values("count", ascending=False).head(30))

In [13]:
rows = []

for dept, idx in df.groupby("department").groups.items():
    dept_counts = np.asarray(X_annotated[list(idx)].sum(axis=0)).ravel()
    top_idx = dept_counts.argsort()[::-1][:TOP_K]

    for k in top_idx:
        rows.append({
            "department": dept,
            "term": terms_annotated[k],
            "count": int(dept_counts[k]),
            "overall": int(overall_annotated[k]),
            "share": round(dept_counts[k] / max(overall_annotated[k], 1), 2),
        })

dept_top_annotated = pd.DataFrame(rows).sort_values(["department", "count"], ascending=[True, False])
dept_top_annotated_f = dept_top_annotated[dept_top_annotated["term"] != "manager"].copy()

In [14]:
p_dept_annotated = (
    ggplot(dept_top_annotated, aes(y=as_discrete("term", order_by="count", order=-1))) +
    geom_bar(aes(x="count", fill="department"), stat="identity") +
    geom_point(aes(x="overall"), size=3) +
    facet_wrap(facets="department", scales="free", ncol=3) +
    #scale_fill_brewer(palette = "Set2") +
    labs(title=f"Label count (bar) + Overall count (dot) - linkedin-cvs-annotated", x="Count", y="Word") +
    theme(legend_position="none") +
    scale_fill_manual(values=DEPT_PALETTE)
    )
p_dept_annotated.show()

In [15]:
p_dept_annotated_f = (
    ggplot(dept_top_annotated_f, aes(y=as_discrete("term", order_by="count", order=-1))) +
    geom_bar(aes(x="count", fill="department"), stat="identity") +
    geom_point(aes(x="overall"), size=3) +
    facet_wrap(facets="department", scales="free", ncol=3) +
    labs(title=f"Label count (bar) + Overall count (dot) - linkedin-cvs-annotated (no 'manager')", x="Count", y="Word") +
    theme(legend_position="none") +
    scale_fill_manual(values=DEPT_PALETTE)
    #scale_fill_brewer(palette = "Set2")
    )
p_dept_annotated_f.show()

In [16]:
#p_dept_annotated_f.to_png("p_dept_annotated_f.png")

In [17]:
label_groups = df.groupby("department", dropna=True).groups
labels = list(label_groups.keys())

label_term_present = np.zeros((len(labels), X_annotated.shape[1]), dtype=bool)

for i, lab in enumerate(labels):
    idx = list(label_groups[lab])
    term_counts = np.asarray(X_annotated[idx].sum(axis=0)).ravel()
    label_term_present[i] = term_counts > 0

term_label_freq = label_term_present.sum(axis=0)

rows = []
for i, lab in enumerate(labels):
    present = label_term_present[i]
    exclusive_mask = present & (term_label_freq == 1)

    rows.append({
        "department": lab,
        "unique_words": int(present.sum()),
        "exclusive_words": int(exclusive_mask.sum()),
    })

rows.append({
    "department": "OVERALL",
    "unique_words": int(X_annotated.shape[1]),
    "exclusive_words": np.nan,
})

unique_words_per_department_annotated = pd.DataFrame(rows).sort_values("unique_words", ascending=False).reset_index(drop=True)

unique_words_per_department_annotated

,department,unique_words,exclusive_words
0,OVERALL,758,NaN
1,Other,543,186.0
2,Information Technology,197,29.0
3,Consulting,169,8.0
4,Sales,160,18.0
5,Project Management,152,14.0
6,Marketing,140,15.0
7,Purchasing,74,13.0
8,Administrative,71,7.0
9,Human Resources,69,9.0


In [18]:
#unique_words_per_department_annotated.to_csv("unique_words_per_department_annotated.csv",index=False)


####**linkedin-cvs-annotated - seniority**

#####Top unigrams per seniority

In [19]:
rows = []

for sen, idx in df.groupby("seniority", dropna=True).groups.items():
    sen_counts = np.asarray(X_annotated[list(idx)].sum(axis=0)).ravel()
    top_idx = sen_counts.argsort()[::-1][:TOP_K]

    for k in top_idx:
        rows.append({
            "seniority": sen,
            "term": terms_annotated[k],
            "count": int(sen_counts[k]),
            "overall": int(overall_annotated[k]),
            "share": round(sen_counts[k] / max(overall_annotated[k], 1), 3),
        })

sen_top_annotated = pd.DataFrame(rows).sort_values(["seniority", "count"], ascending=[True, False])


In [20]:
p_sen_annotated = (
    ggplot(sen_top_annotated, aes(y=as_discrete("term", order_by="count", order=-1))) +
    geom_bar(aes(x="count", fill="seniority"), stat="identity") +
    geom_point(aes(x="overall"), size=3) +
    facet_wrap(facets="seniority", scales="free", ncol=3) +
    labs(title=f"Seniority count (bar) + Overall count (dot) - linkedin-cvs-annotated", x="Count", y="Word") +
    theme(legend_position="none") +
    scale_fill_manual(values=SENIORITY_PALETTE)
    #scale_fill_brewer(palette = "Set2") +
    #ggsize(1600, 1600)
)
p_sen_annotated.show()


In [21]:
#p_sen_annotated.to_png("p_sen_annotated.png")

In [22]:
label_groups = df.groupby("seniority", dropna=True).groups
labels = list(label_groups.keys())

label_term_present = np.zeros((len(labels), X_annotated.shape[1]), dtype=bool)

for i, lab in enumerate(labels):
    idx = list(label_groups[lab])
    term_counts = np.asarray(X_annotated[idx].sum(axis=0)).ravel()
    label_term_present[i] = term_counts > 0

term_label_freq = label_term_present.sum(axis=0)

rows = []
for i, lab in enumerate(labels):
    present = label_term_present[i]
    exclusive_mask = present & (term_label_freq == 1)

    rows.append({
        "seniority": lab,
        "unique_words": int(present.sum()),
        "exclusive_words": int(exclusive_mask.sum()),
    })

rows.append({
    "seniority": "OVERALL",
    "unique_words": int(X_annotated.shape[1]),
    "exclusive_words": np.nan,
})

unique_words_per_seniority_annotated = pd.DataFrame(rows).sort_values("unique_words", ascending=False).reset_index(drop=True)

unique_words_per_seniority_annotated



,seniority,unique_words,exclusive_words
0,OVERALL,758,NaN
1,Professional,545,169.0
2,Lead,367,62.0
3,Junior,177,25.0
4,Management,176,37.0
5,Senior,160,6.0
6,Director,134,11.0


In [23]:
#unique_words_per_seniority_annotated.to_csv("unique_words_per_seniority_annotated.csv",index=False)

####**seniority_v2**

#####Top unigrams per seniority

In [24]:
NGRAMS = (1, 1)

df = df_seniority.copy()
df["pos"] = df["text"]

STOP = []
STOP = ["and", "of", "in", "for", "der", "und", "für"]

vec_seniority_v2 = CountVectorizer(lowercase=True, min_df=1, ngram_range=NGRAMS, stop_words=STOP)
X_seniority_v2 = vec_seniority_v2.fit_transform(df["pos"])

terms_seniority_v2 = np.array(vec_seniority_v2.get_feature_names_out())
overall_seniority_v2 = np.asarray(X_seniority_v2.sum(axis=0)).ravel()

words_top_seniority_v2 = (pd.DataFrame({"term": terms_seniority_v2, "count": overall_seniority_v2}).sort_values("count", ascending=False).head(30))

In [25]:
rows = []

for sen, idx in df.groupby("label", dropna=True).groups.items():
    sen_counts = np.asarray(X_seniority_v2[list(idx)].sum(axis=0)).ravel()
    top_idx = sen_counts.argsort()[::-1][:TOP_K]

    for k in top_idx:
        rows.append({
            "seniority": sen,
            "term": terms_seniority_v2[k],
            "count": int(sen_counts[k]),
            "overall": int(overall_seniority_v2[k]),
            "share": round(sen_counts[k] / max(overall_seniority_v2[k], 1), 3),
        })

sen_top_seniority_v2 = pd.DataFrame(rows).sort_values(["seniority", "count"], ascending=[True, False])


In [26]:
p_sen_seniority_v2  = (
    ggplot(sen_top_seniority_v2, aes(y=as_discrete("term", order_by="count", order=-1))) +
    geom_bar(aes(x="count", fill="seniority"), stat="identity") +
    geom_point(aes(x="overall"), size=3) +
    facet_wrap(facets="seniority", scales="free", ncol=3) +
    labs(title=f"Label count (bar) + Overall count (dot) - seniority-v2", x="Count", y="Word") +
    theme(legend_position="none") +
    scale_fill_manual(values=SENIORITY_PALETTE)
    #scale_fill_brewer(palette = "Set2") +
    #ggsize(1600, 1600)
)
p_sen_seniority_v2 .show()

In [27]:
#p_sen_seniority_v2 .to_png("p_sen_seniority_v2 .png")

In [28]:
label_groups = df.groupby("label", dropna=True).groups
labels = list(label_groups.keys())

label_term_present = np.zeros((len(labels), X_seniority_v2.shape[1]), dtype=bool)

for i, lab in enumerate(labels):
    idx = list(label_groups[lab])
    term_counts = np.asarray(X_seniority_v2[idx].sum(axis=0)).ravel()
    label_term_present[i] = term_counts > 0

term_label_freq = label_term_present.sum(axis=0)

rows = []
for i, lab in enumerate(labels):
    present = label_term_present[i]
    exclusive_mask = present & (term_label_freq == 1)

    rows.append({
        "seniority": lab,
        "unique_words": int(present.sum()),
        "exclusive_words": int(exclusive_mask.sum()),
    })

rows.append({
    "seniority": "OVERALL",
    "unique_words": int(X_seniority_v2.shape[1]),
    "exclusive_words": np.nan,
})

unique_words_per_seniority_seniority_v2 = pd.DataFrame(rows).sort_values("unique_words", ascending=False).reset_index(drop=True)

unique_words_per_seniority_seniority_v2



,seniority,unique_words,exclusive_words
0,OVERALL,3340,NaN
1,Lead,1854,1118.0
2,Senior,1631,867.0
3,Director,630,210.0
4,Management,538,209.0
5,Junior,292,64.0


In [29]:
#unique_words_per_seniority_seniority_v2.to_csv("unique_words_per_seniority_seniority_v2.csv",index=False)

####**department_v2**

#####Top unigrams per department

In [30]:
NGRAMS = (1, 1)

df = df_department.copy()
df["pos"] = df["text"]

STOP = []
STOP = ["and", "of", "in", "for", "der", "und", "für", "des"]

vec_department_v2 = CountVectorizer(lowercase=True, min_df=1, ngram_range=NGRAMS, stop_words=STOP)
X_department_v2 = vec_department_v2.fit_transform(df["pos"])

terms_department_v2 = np.array(vec_department_v2.get_feature_names_out())
overall_department_v2 = np.asarray(X_department_v2.sum(axis=0)).ravel()

words_top_department_v2 = (pd.DataFrame({"term": terms_department_v2, "count": overall_department_v2}).sort_values("count", ascending=False).head(30))


In [31]:
rows = []

for dept, idx in df.groupby("label").groups.items():
    dept_counts = np.asarray(X_department_v2[list(idx)].sum(axis=0)).ravel()
    top_idx = dept_counts.argsort()[::-1][:TOP_K]

    for k in top_idx:
        rows.append({
            "department": dept,
            "term": terms_department_v2[k],
            "count": int(dept_counts[k]),
            "overall": int(overall_department_v2[k]),
            "share": round(dept_counts[k] / max(overall_department_v2[k], 1), 2),
        })

dept_top_department_v2 = pd.DataFrame(rows).sort_values(["department", "count"], ascending=[True, False])
dept_top_department_v2_f = dept_top_department_v2[dept_top_department_v2["term"] != "manager"].copy()


In [32]:
p_dept_department_v2 = (
    ggplot(dept_top_department_v2, aes(y=as_discrete("term", order_by="count", order=-1))) +
    geom_bar(aes(x="count", fill="department"), stat="identity") +
    geom_point(aes(x="overall"), size=3) +
    facet_wrap(facets="department", scales="free", ncol=3) +
    labs(title=f"Label count (bar) + Overall count (dot) - department-v2", x="Count", y="Word") +
    theme(legend_position="none") +
    scale_fill_manual(values=DEPT_PALETTE)
    #scale_fill_brewer(palette = "Set2")
    )
p_dept_department_v2.show()

In [33]:
p_dept_department_v2_f = (
    ggplot(dept_top_department_v2_f, aes(y=as_discrete("term", order_by="count", order=-1))) +
    geom_bar(aes(x="count", fill="department"), stat="identity") +
    geom_point(aes(x="overall"), size=3) +
    facet_wrap(facets="department", scales="free", ncol=3) +
    labs(title=f"Label count (bar) + Overall count (dot) - department-v2 (no 'manager')", x="Count", y="Word") +
    theme(legend_position="none") +
    scale_fill_manual(values=DEPT_PALETTE)
    #scale_fill_brewer(palette = "Set2")
    )
p_dept_department_v2_f.show()

In [34]:
#p_dept_department_v2_f.to_png("p_dept_department_v2_f.png")

In [35]:
label_groups = df.groupby("label", dropna=True).groups

labels = list(label_groups.keys())
label_term_present = np.zeros((len(labels), X_department_v2.shape[1]), dtype=bool)

for i, lab in enumerate(labels):
    idx = list(label_groups[lab])
    term_counts = np.asarray(X_department_v2[idx].sum(axis=0)).ravel()
    label_term_present[i] = term_counts > 0

term_label_freq = label_term_present.sum(axis=0)

rows = []
for i, lab in enumerate(labels):
    present = label_term_present[i]


    unique_words = int(present.sum())


    exclusive_mask = present & (term_label_freq == 1)
    exclusive_words = exclusive_mask.sum()


    rows.append({
        "department": lab,
        "unique_words": unique_words,
        "exclusive_words": exclusive_words,
    })

rows.append({
    "department": "OVERALL",
    "unique_words": int(X_department_v2.shape[1]),
    "exclusive_words": np.nan,
})

unique_words_per_department_department_v2 = pd.DataFrame(rows).sort_values("unique_words", ascending=False).reset_index(drop=True)

unique_words_per_department_department_v2

,department,unique_words,exclusive_words
0,OVERALL,3397,NaN
1,Sales,1758,989.0
2,Marketing,1624,870.0
3,Information Technology,840,309.0
4,Business Development,444,117.0
5,Project Management,270,60.0
6,Consulting,193,45.0
7,Administrative,76,22.0
8,Customer Support,73,11.0
9,Other,52,5.0


In [36]:
#unique_words_per_department_department_v2.to_csv("unique_words_per_department_department_v2.csv",index=False)

In [37]:
TOP_N = 10
MAX_ROWS_PER_LABEL = 100
DROP = {"and", "of", "der", "und", "für", "des"}

label_sizes = df.groupby("label").size()
small_labels = set(label_sizes[label_sizes < MAX_ROWS_PER_LABEL].index)

rows = []
for dept, idx in df.groupby("label").groups.items():
    if dept not in small_labels:
        continue

    dept_counts = np.asarray(X_department_v2[list(idx)].sum(axis=0)).ravel()

    filtered = dept_counts.copy()
    filtered[np.isin(terms_department_v2, list(DROP))] = 0

    top_idx = filtered.argsort()[::-1][:TOP_N]

    for k in top_idx:
        if dept_counts[k] == 0:
            continue
        rows.append({
            "department": dept,
            "term": terms_department_v2[k],
            "count": int(dept_counts[k]),
            "overall": int(overall_department_v2[k]),
            "share": round(dept_counts[k] / max(overall_department_v2[k], 1), 2),
        })

dept_top_small_labels = pd.DataFrame(rows).sort_values(["department", "count"], ascending=[True, False])



In [38]:
dept_top_small_labels = dept_top_small_labels.copy()
dept_top_small_labels["overall_cap100"] = dept_top_small_labels["overall"].clip(upper=75)

p_dept = (
    ggplot(dept_top_small_labels, aes(y=as_discrete("term", order_by="count", order=-1))) +
    geom_bar(aes(x="count", fill="department"), stat="identity") +
    geom_point(aes(x="overall_cap100"), size=3) +
    facet_wrap(facets="department", scales="free", ncol=3) +
    labs(title="Label count (bar) + Overall count (dot) - department-v2", x="Count", y="Word") +
    theme(legend_position="none") +
    scale_fill_manual(values=DEPT_PALETTE)
)

p_dept

In [48]:
#p_dept.to_png("p_dept.png")

'/content/p_dept.png'

In [40]:
TOP_N = 10
MIN_ROWS_PER_LABEL = 100

label_sizes = df.groupby("label").size()
big_labels = set(label_sizes[label_sizes >= MIN_ROWS_PER_LABEL].index)

rows = []
for dept, idx in df.groupby("label").groups.items():
    if dept not in big_labels:
        continue

    dept_counts = np.asarray(X_department_v2[list(idx)].sum(axis=0)).ravel()
    top_idx = dept_counts.argsort()[::-1][:TOP_N]

    for k in top_idx:
        if dept_counts[k] == 0:
            continue
        rows.append({
            "department": dept,
            "term": terms_department_v2[k],
            "count": int(dept_counts[k]),
            "overall": int(overall_department_v2[k]),
            "share": round(dept_counts[k] / max(overall_department_v2[k], 1), 2),
        })

dept_top_big = pd.DataFrame(rows).sort_values(["department", "count"], ascending=[True, False])

dept_top_big = dept_top_big[dept_top_big["term"] != "manager"].copy()

p_dept_2 = (
    ggplot(dept_top_big, aes(y=as_discrete("term", order_by="count", order=-1))) +
    geom_bar(aes(x="count", fill="department"), stat="identity") +
    geom_point(aes(x="overall"), size=3) +
    facet_wrap(facets="department", scales="free", ncol=3) +
    labs(title="Label count (bar) + Overall count (dot) - department-v2 (no 'manager')", x="Count", y="Word") +
    theme(legend_position="none") +
    scale_fill_manual(values=DEPT_PALETTE)
)

p_dept_2


In [47]:
#p_dept_2.to_png("p_dept_2.png")

'/content/p_dept_2.png'

####**overall vocabulary**

comparing term frequencies / shares between datasets

In [42]:
NGRAMS = (1, 1)

df = df_t_notannotated.copy()
df["pos"] = df["position"]

STOP = []
STOP = ["and", "of", "in", "for", "der", "und", "für"]

vec_not_annotated = CountVectorizer(lowercase=True, min_df=2, ngram_range=NGRAMS, stop_words=STOP)
X_not_annotated = vec_not_annotated.fit_transform(df["pos"])

terms_not_annotated = np.array(vec_not_annotated.get_feature_names_out())
overall_not_annotated = np.asarray(X_not_annotated.sum(axis=0)).ravel()

words_top_not_annotated = (pd.DataFrame({"term": terms_not_annotated, "count": overall_not_annotated}).sort_values("count", ascending=False).head(30))

In [43]:
print("linkedin-cvs-annotated")
print("--------------------------------------------------------")
print(words_top_annotated)

print("\n")
print("linkedin-cvs-not-annotated")
print("--------------------------------------------------------")
print(words_top_not_annotated)

linkedin-cvs-annotated
--------------------------------------------------------
                term  count
427          manager    395
209         director    166
616           senior    160
156       consultant    112
325             head    107
100         business     97
593            sales     97
539          project     80
240         engineer     76
426       management     75
376               it     66
202      development     61
4            account     59
308  geschäftsführer     57
433        marketing     57
40           analyst     53
57         assistant     52
520        president     44
429         managing     44
260        executive     44
637       specialist     43
475          officer     38
187               de     38
411           leiter     37
528          product     37
582         research     37
495          partner     37
730             vice     36
629         software     36
619          service     36


linkedin-cvs-not-annotated
-----------------------

In [44]:
TOP_K = 11
NGRAMS = (1, 1)

df = df_t_annotated.copy()
df["pos"] = df["position"]

STOP = []
#STOP = ["and", "of", "in", "for", "der", "und", "für"]

vec_annotated = CountVectorizer(lowercase=True, min_df=2, ngram_range=NGRAMS, stop_words=STOP)
X_annotated = vec_annotated.fit_transform(df["pos"])

In [45]:
linkedin_cvs_annotated_vocab = set(vec_annotated.get_feature_names_out())
analyze = vec_annotated.build_analyzer()

department_v2_vocab = set(analyze(" ".join(df_department["text"])))

print("linkedin_cvs_annotated_vocab:", len(linkedin_cvs_annotated_vocab))
print("department_v2_vocab:", len(department_v2_vocab))
print("overlap:", len(linkedin_cvs_annotated_vocab & department_v2_vocab))
print("only in linkedin-cvs-annotated:", len(linkedin_cvs_annotated_vocab - department_v2_vocab))
print("only in department-v2:", len(department_v2_vocab - linkedin_cvs_annotated_vocab))


seniority_v2_vocab = set(analyze(" ".join(df_seniority["text"])))

print("\nlinkedin_cvs_annotated_vocab:", len(linkedin_cvs_annotated_vocab))
print("seniority_v2_vocab:", len(seniority_v2_vocab))
print("overlap:", len(linkedin_cvs_annotated_vocab & seniority_v2_vocab))
print("only in linkedin-cvs-annotated:", len(linkedin_cvs_annotated_vocab - seniority_v2_vocab))
print("only in seniority-v2:", len(seniority_v2_vocab - linkedin_cvs_annotated_vocab))


linkedin_cvs_annotated_vocab: 765
department_v2_vocab: 3405
overlap: 466
only in linkedin-cvs-annotated: 299
only in department-v2: 2939

linkedin_cvs_annotated_vocab: 765
seniority_v2_vocab: 3347
overlap: 477
only in linkedin-cvs-annotated: 288
only in seniority-v2: 2870


Most common position (whole value)

In [46]:
position_counts_annotated = df_t_annotated["position"].value_counts()

print("linkedin-cvs-annotated")
print("--------------------------------------------------------")
print(position_counts_annotated.head(25))

print("\n")
print("linkedin-cvs-not-annotated")
print("--------------------------------------------------------")
position_counts_notannotated = df_t_notannotated["position"].value_counts()

print(position_counts_notannotated.head(25))

linkedin-cvs-annotated
--------------------------------------------------------
position
Geschäftsführer                      41
Managing Director                    22
CEO                                  20
Consultant                           18
Project Manager                      16
Owner                                12
Vice President                       11
Senior Project Manager               11
Director                             11
Partner                              10
General Manager                      10
Unternehmensinhaber                   9
Geschäftsführender Gesellschafter     9
Senior Consultant                     9
Rechtsanwalt                          8
Professor                             8
Intern                                7
Account Manager                       7
Sales Manager                         7
CFO                                   7
Key-Account-Manager                   7
Human Resources Manager               7
Software Developer             

##date/status/seniority/department

In [ ]:
print("linkedin-cvs-annotated")
df_t_annotated["startDate"] = pd.to_datetime(df_t_annotated["startDate"], errors="coerce")
df_t_annotated["endDate"] = pd.to_datetime(df_t_annotated["endDate"], errors="coerce")

missing_start = df_t_annotated["startDate"].isna().sum()
missing_end = df_t_annotated["endDate"].isna().sum()
total_rows = len(df_t_annotated)

print(f"Total rows: {total_rows}")
print(f"Missing startDate: {missing_start}")
print(f"Missing endDate: {missing_end}")

df_t_annotated["duration_days"] = (df_t_annotated["endDate"] - df_t_annotated["startDate"]).dt.days

duration_summary = df_t_annotated["duration_days"].agg(["min", "max", "mean", "median"]).reset_index()
duration_summary.columns = ["statistic", "value"]

print("\nDuration (days) Summary:")
print(duration_summary)

seniority_duration = df_t_annotated.dropna(subset=["duration_days", "seniority"]).groupby("seniority")["duration_days"].agg(["count", "median", "mean", "min", "max"]).reset_index()

print("\nSeniority vs Duration Summary:")
print(seniority_duration)

department_duration = df_t_annotated.dropna(subset=["duration_days", "department"]).groupby("department")["duration_days"].agg(["count", "median", "mean", "min", "max"]).reset_index()

print("\nDepartment vs Duration Summary:")
print(department_duration)

linkedin-cvs-annotated
Total rows: 2638
Missing startDate: 334
Missing endDate: 924

Duration (days) Summary:
  statistic         value
0       min      0.000000
1       max  14610.000000
2      mean   1339.563936
3    median    853.000000

Seniority vs Duration Summary:
      seniority  count  median         mean   min      max
0      Director     95  1095.0  1666.842105  60.0   8036.0
1        Junior    204   275.0   578.411765   0.0   8249.0
2          Lead    289  1006.0  1583.910035  30.0  11841.0
3    Management    169  1430.0  1995.047337  89.0  14610.0
4  Professional    828   822.0  1245.170290   0.0  13028.0
5        Senior    112   912.5  1526.607143  92.0  10043.0

Department vs Duration Summary:
                department  count  median         mean    min      max
0           Administrative     59   518.0   846.101695    0.0   5508.0
1     Business Development     47   883.0  1574.872340  184.0  10043.0
2               Consulting    126   532.5   998.055556   31.0   6909.